# all-reduce-compose — faded example 1: Complete the MIN all_reduce composition

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `all-reduce-compose`. The last cell reports your progress on the `Distributed: all_reduce composition` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: all_reduce composition` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`all-reduce-compose`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "all-reduce-compose"
DD_SUBTOPIC = "Distributed: all_reduce composition"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

An all_reduce with the MIN op gives every rank the global minimum across all rank-local values. It is the same `reduce(dst=0)` then `broadcast(src=0)` composition as SUM/MAX, only the `ReduceOp` changes. The reduce collapses the minimum onto rank 0; the broadcast spreads it to all ranks.

## Faded exercise 1

### Faded — finish the MIN all_reduce

The worker `ex_all_reduce_min(rank, world_size, dist_module, local_value)` should leave every rank holding the global **minimum** of the rank-local values. The tensor construction, the broadcast, and the return are already written. Complete the single reduce call that collapses the minimum onto rank 0.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import threading

class MockDist:
    class ReduceOp:
        SUM = 'sum'
        MAX = 'max'
        MIN = 'min'
    def __init__(self, world_size):
        self.world_size = world_size
        self._barrier = threading.Barrier(world_size)
        self._slots = [None] * world_size
        self._result = [None]
    def reduce(self, tensor, dst, op):
        rank = threading.current_thread().rank
        self._slots[rank] = tensor.clone()
        self._barrier.wait()
        if rank == dst:
            vals = t.stack(self._slots)
            if op == self.ReduceOp.SUM:
                tensor.copy_(vals.sum(dim=0))
            elif op == self.ReduceOp.MAX:
                tensor.copy_(vals.amax(dim=0))
            elif op == self.ReduceOp.MIN:
                tensor.copy_(vals.amin(dim=0))
            self._result[0] = tensor.clone()
        self._barrier.wait()
    def broadcast(self, tensor, src):
        self._barrier.wait()
        tensor.copy_(self._result[0])
        self._barrier.wait()

def ex_all_reduce_min(rank: int, world_size: int, dist_module, local_value: float) -> float:
    tensor = t.tensor([local_value], dtype=t.float32)
    raise NotImplementedError()  # TODO: fill in this step — read the prompt cell above
    dist_module.broadcast(tensor, src=0)
    return tensor.item()

def _test():
    world_size = 4
    locals_ = [9.0, 3.0, 7.0, 5.0]
    mock = MockDist(world_size)
    results = [None] * world_size
    def _run(rank):
        threading.current_thread().rank = rank
        results[rank] = ex_all_reduce_min(rank, world_size, mock, locals_[rank])
    threads = [threading.Thread(target=_run, args=(r,)) for r in range(world_size)]
    for th in threads: th.start()
    for th in threads: th.join()
    expected = min(locals_)
    assert len(set(results)) == 1, f'ranks disagree: {results}'
    assert abs(results[0] - expected) < 1e-6, f'got {results[0]}, expected {expected}'

try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import threading

class MockDist:
    class ReduceOp:
        SUM = 'sum'
        MAX = 'max'
        MIN = 'min'
    def __init__(self, world_size):
        self.world_size = world_size
        self._barrier = threading.Barrier(world_size)
        self._slots = [None] * world_size
        self._result = [None]
    def reduce(self, tensor, dst, op):
        rank = threading.current_thread().rank
        self._slots[rank] = tensor.clone()
        self._barrier.wait()
        if rank == dst:
            vals = t.stack(self._slots)
            if op == self.ReduceOp.SUM:
                tensor.copy_(vals.sum(dim=0))
            elif op == self.ReduceOp.MAX:
                tensor.copy_(vals.amax(dim=0))
            elif op == self.ReduceOp.MIN:
                tensor.copy_(vals.amin(dim=0))
            self._result[0] = tensor.clone()
        self._barrier.wait()
    def broadcast(self, tensor, src):
        self._barrier.wait()
        tensor.copy_(self._result[0])
        self._barrier.wait()

def ex_all_reduce_min(rank: int, world_size: int, dist_module, local_value: float) -> float:
    tensor = t.tensor([local_value], dtype=t.float32)
    dist_module.reduce(tensor, dst=0, op=dist_module.ReduceOp.MIN)
    dist_module.broadcast(tensor, src=0)
    return tensor.item()
```
</details>